# Random Forest

#### Objetivo
- Construir um pipeline RandomForest para prever “Reprovou” (classe minoritária = 1).
- Priorizar recall da classe 1 controlando falsos positivos via seleção de threshold validada.
- Splits estratificados: 70% treino, 20% teste, 10% validação.
- StratifiedKFold + GridSearchCV com grade compacta.
- Threshold robusto via K-fold + refinamento no hold-out.

## 1. Preparação do notebook

Primeiramente, realizou-se a configuração do notebook para o treino e teste do modelo random forest.

### 1.1. Importações e Configuração

Esta célula inicial concentra a importação de todas as bibliotecas necessárias e a definição das configurações globais que guiarão o experimento. Estabelecemos constantes para os caminhos dos dados, a variável alvo (`Reprovou`), e a semente de aleatoriedade (`RANDOM_STATE`) para garantir a reprodutibilidade dos resultados. Além disso, definimos os parâmetros fundamentais para a modelagem, como a meta de recall para a classe minoritária, a grade de hiperparâmetros para o `RandomForestClassifier`, a estratégia de validação cruzada estratificada (`StratifiedKFold`) e o scorer customizado que será utilizado para otimizar o modelo durante a busca em grade.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
 make_scorer, recall_score, precision_score, f1_score,
 accuracy_score, classification_report, confusion_matrix
)

# Configurações gerais e seeds
DATA_DIR = Path('dados')
CSV_PATHS = {
 1: DATA_DIR/'dados_modelo1.csv',
 2: DATA_DIR/'dados_modelo2.csv',
 3: DATA_DIR/'dados_modelo3.csv'
}
TARGET = 'Reprovou'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Classe minoritária e meta de recall
MINORITY_LABEL = 1 # Reprovou
RECALL_MIN_TARGET = 0.50

# Preferência para desempate de threshold (entre candidatos que atingem o recall)
THRESHOLD_PREFERENCE = 'f1' # 'precision' ou 'f1'

# Margem aplicada ao threshold baseado na mediana dos folds (favorece recall)
THRESHOLD_SHIFT = 0.05

# Parâmetros base do RF
RF_BASE_PARAMS = dict(
 n_estimators=400,
 random_state=RANDOM_STATE,
 n_jobs=-1,
 class_weight='balanced'
)

# Grade de hiperparâmetros compacta (inclui variação de class_weight)
PARAM_GRID = {
 'rf__max_depth': [4, 8, None],
 'rf__min_samples_leaf': [3, 8],
 'rf__min_samples_split': [5, 15],
 'rf__max_features': ['sqrt', 0.5],
 'rf__class_weight': ['balanced', {0: 1.0, 1: 2.0}],
}

# Validação cruzada estratificada
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Scorer para maximizar recall da classe minoritária
RECALL_MINORITY_SCORER = make_scorer(recall_score, pos_label=MINORITY_LABEL, zero_division=0)

## 2. Treino do modelo

Com o ambiente e os parâmetros devidamente configurados, iniciamos a etapa de treinamento do modelo. Nesta seção, aplicaremos uma busca em grade com validação cruzada para identificar a combinação de hiperparâmetros que melhor generaliza para novos dados, focando em nosso objetivo principal de maximizar o recall da classe minoritária. O processo resultará em um pipeline treinado e otimizado para a tarefa de predição.

### 2.1. Pipeline de Pré-processamento

Para garantir que o pré-processamento seja aplicado de forma consistente durante a validação cruzada, definimos duas funções para construir nosso pipeline. A função build_preprocess cria um ColumnTransformer que aplica imputação pela mediana a colunas numéricas e uma sequência de imputação pela moda seguida de OneHotEncoder a colunas categóricas. Em seguida, a função make_pipeline_for_grid integra este pré-processador a um estimador RandomForestClassifier, gerando um pipeline completo e pronto para ser otimizado pelo GridSearchCV.

In [7]:
def build_preprocess(X: pd.DataFrame) -> ColumnTransformer:
 num_cols = X.select_dtypes(include=['number']).columns.tolist()
 cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
 return ColumnTransformer([
 ('num', SimpleImputer(strategy='median'), num_cols),
 ('cat', Pipeline([
 ('imp', SimpleImputer(strategy='most_frequent')),
 ('ohe', OneHotEncoder(handle_unknown='ignore'))
 ]), cat_cols)
 ])

def make_pipeline_for_grid(X: pd.DataFrame) -> Pipeline:
 return Pipeline([('prep', build_preprocess(X)), ('rf', RandomForestClassifier(**RF_BASE_PARAMS))])

### 2.2 Seleção de Threshold

O modelo gera uma probabilidade de um aluno reprovar, e precisamos escolher um valor de corte (threshold) para tomar a decisão final, em vez de usar o padrão 0.5. Para isso, criamos três funções. A primeira, `eval_thresholds`, testa vários valores e escolhe o melhor que atinge nossa meta de recall. A segunda, `choose_threshold_cv`, repete esse teste várias vezes nos dados de treino para achar um threshold mediano mais estável. A terceira, `refine_threshold_around_median`, usa esse valor mediano como base para fazer um ajuste final e encontrar o melhor threshold possível nos dados de validação.

In [8]:
def eval_thresholds(proba_min: np.ndarray, y_true: pd.Series, pref: str, rec_target: float):
    thresholds = np.linspace(0, 1, 201)
    stats = []
    for thr in thresholds:
        y_pred = (proba_min >= thr).astype(int)  # 1 (minoritária) se >= thr
        rec = recall_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        prec = precision_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        f1 = f1_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        stats.append((thr, rec, prec, f1))
    feasible = [s for s in stats if s[1] >= rec_target] or sorted(stats, key=lambda x: (x[1], x[3], x[2]), reverse=True)[:5]
    key = (lambda x: (x[2], x[1], x[3])) if pref == 'precision' else (lambda x: (x[3], x[1], x[2]))
    return max(feasible, key=key)  # (thr, rec, prec, f1)

def choose_threshold_cv(pipe: Pipeline, X_tr: pd.DataFrame, y_tr: pd.Series,
    pref: str, rec_target: float, cv: StratifiedKFold, shift: float):
    thrs = []
    X_tr = X_tr.reset_index(drop=True)
    y_tr = y_tr.reset_index(drop=True)
    for tr_idx, va_idx in cv.split(X_tr, y_tr):
        pipe_fold = Pipeline(pipe.steps)  # shallow copy
        pipe_fold.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx])
        idx_min = list(pipe_fold.named_steps['rf'].classes_).index(MINORITY_LABEL)
        proba_min = pipe_fold.predict_proba(X_tr.iloc[va_idx])[:, idx_min]
        thr, *_ = eval_thresholds(proba_min, y_tr.iloc[va_idx], pref, rec_target)
        thrs.append(thr)
    median_thr = float(np.median(thrs))
    thr_base = max(0.0, median_thr - shift)
    return thr_base, thrs, median_thr

def refine_threshold_around_median(pipe: Pipeline, X_val: pd.DataFrame, y_val: pd.Series,
    median_thr: float, pref: str, rec_target: float,
    window: float = 0.15, step: float = 0.01):
    idx_min = list(pipe.named_steps['rf'].classes_).index(MINORITY_LABEL)
    proba_val = pipe.predict_proba(X_val)[:, idx_min]
    thrs = np.clip(np.arange(median_thr - window, median_thr + window + step, step), 0, 1)
    cand = []
    for thr in thrs:
        y_pred = (proba_val >= thr).astype(int)
        rec = recall_score(y_val, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        prec = precision_score(y_val, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        f1 = f1_score(y_val, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        cand.append((thr, rec, prec, f1))
    feasible = [c for c in cand if c[1] >= rec_target] or sorted(cand, key=lambda x: (x[1], x[3], x[2]), reverse=True)[:5]
    key = (lambda x: (x[2], x[1], x[3])) if pref == 'precision' else (lambda x: (x[3], x[1], x[2]))
    return max(feasible, key=key)[0]

### 2.3. Métricas e Busca de Hiperparâmetros

Estas são as funções que utilizaremos para avaliar e treinar o modelo. A função `metrics_dict` calcula de uma só vez as métricas de performance mais importantes (acurácia, recall, precisão e F1-score) para a classe de interesse. A `run_grid` executa a busca `GridSearchCV`, que testa sistematicamente várias combinações de hiperparâmetros para encontrar a que resulta no melhor modelo, usando o recall como critério de otimização. Por último, a `resumir_params` é uma pequena função auxiliar para exibir os melhores parâmetros encontrados de forma limpa e resumida.

In [9]:
def metrics_dict(y_true, y_pred):
 return dict(
 accuracy=accuracy_score(y_true, y_pred),
 recall_min=recall_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0),
 precision_min=precision_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0),
 f1_min=f1_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0),
 )

def run_grid(X_tr: pd.DataFrame, y_tr: pd.Series):
 grid = GridSearchCV(
 estimator=make_pipeline_for_grid(X_tr),
 param_grid=PARAM_GRID,
 scoring=RECALL_MINORITY_SCORER,
 cv=CV,
 n_jobs=-1,
 refit=True,
 verbose=0
 )
 grid.fit(X_tr, y_tr)
 return grid.best_estimator_, grid.best_params_, grid.best_score_

def resumir_params(best_params: dict) -> str:
 mp = lambda k: best_params.get(f'rf__{k}')
 return f"depth={mp('max_depth')}, msl={mp('min_samples_leaf')}, mss={mp('min_samples_split')}, mf={mp('max_features')}"

### 2.4 Treino, Seleção de Threshold e Avaliação

Este é o bloco principal onde executamos todo o processo de forma iterativa para cada conjunto de dados definido. Primeiramente, os dados são carregados e divididos em três partes: treino (70%), validação (10%) e teste (20%). Em seguida, realizamos a busca pelos melhores hiperparâmetros, selecionamos o threshold de decisão ótimo usando os dados de treino e validação, e, por fim, avaliamos o desempenho do modelo final no conjunto de teste, que permaneceu intocado até este momento. Todos os resultados, métricas e parâmetros são impressos e armazenados para análise posterior.

In [10]:
all_rows = []

for table_id, csv_path in CSV_PATHS.items():
    print(f"\n======================\nTabela {table_id}: {csv_path}\n======================")
    if not csv_path.exists():
        print(f"Arquivo não encontrado: {csv_path}. Pulando.")
        continue

    df = pd.read_csv(csv_path)
    if TARGET not in df.columns:
        print(f"Coluna alvo '{TARGET}' não encontrada. Pulando.")
        continue

    y = df[TARGET].astype(int)
    X = df.drop(columns=[TARGET])

    # Splits estratificados: 70/20/10
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
    X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=1/3, stratify=y_temp, random_state=RANDOM_STATE)
    print(f"Formas: treino={X_train.shape}, teste={X_test.shape}, val={X_val.shape}")

    # 1) GridSearchCV maximizando recall(1)
    best_pipe, best_params, best_cv_recall = run_grid(X_train, y_train)

    # 2) Threshold robusto via K-fold no treino
    thr_base, thr_list, median_thr = choose_threshold_cv(
        best_pipe, X_train, y_train,
        pref=THRESHOLD_PREFERENCE, rec_target=RECALL_MIN_TARGET,
        cv=CV, shift=THRESHOLD_SHIFT
    )

    # 2b) Refinar threshold no hold-out (melhora precisão controlando recall)
    thr_refined = refine_threshold_around_median(
        best_pipe, X_val, y_val, median_thr, THRESHOLD_PREFERENCE, RECALL_MIN_TARGET,
        window=0.15, step=0.01
    )

    # Threshold final usado
    thr_used = float(thr_refined)

    # 3) Avaliação final em Val e Teste
    idx_min = list(best_pipe.named_steps['rf'].classes_).index(MINORITY_LABEL)
    proba_val = best_pipe.predict_proba(X_val)[:, idx_min]
    proba_test = best_pipe.predict_proba(X_test)[:, idx_min]
    y_val_pred = (proba_val >= thr_used).astype(int)
    y_test_pred = (proba_test >= thr_used).astype(int)

    # Métricas
    val_rec = recall_score(y_val, y_val_pred, pos_label=MINORITY_LABEL, zero_division=0)
    val_prec = precision_score(y_val, y_val_pred, pos_label=MINORITY_LABEL, zero_division=0)
    val_f1 = f1_score(y_val, y_val_pred, pos_label=MINORITY_LABEL, zero_division=0)
    m = metrics_dict(y_test, y_test_pred)

    # Logs compactos
    print(f"[Tabela {table_id}] thr_base={thr_base:.3f} (mediana={median_thr:.3f}, shift={THRESHOLD_SHIFT}) | thr_refined={thr_used:.3f}")
    print(f"Val (thr): recall_min={val_rec:.3f}, precision_min={val_prec:.3f}, f1_min={val_f1:.3f}")
    print(f"Teste: Acc={m['accuracy']:.4f}, Recall_min={m['recall_min']:.4f}, Prec_min={m['precision_min']:.4f}, F1_min={m['f1_min']:.4f}")

    # Relatórios legíveis
    print("\nRelatório por classe (teste):")
    print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))
    print("Matriz de confusão (teste):")
    print(confusion_matrix(y_test, y_test_pred))

    all_rows.append({
        'tabela_id': table_id,
        'thr_cv': round(thr_used, 3),
        'recall_min': round(m['recall_min'], 3),
        'precision_min': round(m['precision_min'], 3),
        'f1_min': round(m['f1_min'], 3),
        'accuracy': round(m['accuracy'], 3),
        'best_params_resumidos': resumir_params(best_params)
    })


Tabela 1: dados/dados_modelo1.csv
Formas: treino=(962, 8), teste=(275, 8), val=(138, 8)


[Tabela 1] thr_base=0.365 (mediana=0.415, shift=0.05) | thr_refined=0.575
Val (thr): recall_min=0.500, precision_min=0.250, f1_min=0.333
Teste: Acc=0.8800, Recall_min=0.4375, Prec_min=0.2258, F1_min=0.2979

Relatório por classe (teste):
              precision    recall  f1-score   support

           0     0.9631    0.9073    0.9344       259
           1     0.2258    0.4375    0.2979        16

    accuracy                         0.8800       275
   macro avg     0.5945    0.6724    0.6161       275
weighted avg     0.9202    0.8800    0.8974       275

Matriz de confusão (teste):
[[235  24]
 [  9   7]]

Tabela 2: dados/dados_modelo2.csv
Formas: treino=(962, 11), teste=(275, 11), val=(138, 11)
[Tabela 2] thr_base=0.510 (mediana=0.560, shift=0.05) | thr_refined=0.410
Val (thr): recall_min=0.750, precision_min=0.429, f1_min=0.545
Teste: Acc=0.8836, Recall_min=0.6250, Prec_min=0.2778, F1_min=0.3846

Relatório por classe (teste):
              precision    recall  f1-score   support

 

### 2.5. Resumo Final  (em arquivo)

Ao final de todas as execuções, esta célula agrupa os resultados mais importantes de cada modelo treinado em uma única tabela. Os dados, que foram armazenados em uma lista, são convertidos para um DataFrame do pandas para facilitar a visualização. O código então exibe este DataFrame resumido, priorizando uma formatação amigável para notebooks, ou imprimindo como texto simples caso o ambiente não suporte a exibição rica.

In [ ]:
if all_rows:
    df_out = pd.DataFrame(all_rows).sort_values('tabela_id')
    cols = ['tabela_id', 'thr_cv', 'recall_min', 'precision_min', 'f1_min', 'accuracy', 'best_params_resumidos']

    try:
        from IPython.display import display
        display(df_out[cols])
    except Exception:

     print(df_out[cols].to_string(index=False))

else:
    print("Nenhum resumo gerado. Verifique os arquivos e a coluna alvo.")

## 3. Conclusões 

Apesar do objetivo de atingir um recall mínimo de 0.50, o modelo Random Forest apresentou resultados gerais pouco satisfatórios e instáveis ao ser aplicado em dados de semanas distintas. A meta só foi alcançada nos dados da Semana 2 (recall de 0.6250), e ainda assim com uma precisão muito baixa, enquanto nas demais semanas o desempenho ficou aquém do esperado. Essa inconsistência de performance, somada ao alto custo de falsos positivos mesmo no melhor cenário, indica que o modelo, em seu estado atual, não demonstra a confiabilidade necessária para uma implementação eficaz.